## Hallucination Detection

<img src="https://live.staticflickr.com/65535/54208132682_73767c3560_b.jpg" alt="Embedded Photo" width="500">

*Image generated using the DALL·E model.*

---

## Introduction

Language models help us with everyday tasks such as proofreading text, writing code, or answering questions.
They are also increasingly used in fields like medicine and education.

However, how can we know whether the answers they generate are correct? Language models do not always have complete knowledge about a given topic, yet they can still produce responses that sound convincing but are actually misleading. Such incorrect responses are called **hallucinations**.

---

## Task

In this task, you will work on detecting hallucinations in answers to factual questions generated by large language models (LLMs).
You will analyze a dataset that helps determine whether the model’s responses are actually correct or contain hallucinations.

Each example in the dataset contains:

* **Question**, e.g., *"What is the main responsibility of the U.S. Department of Defense?"*
* **Model answer**, e.g., *"The main responsibility is to defend the country."*
* **Tokens** associated with the generated answer
* **Four alternative answers** generated by the same model with higher temperature
* **Tokens of the alternative answers**
* **Probabilities of the alternative answers**
* **Label (`is_correct`)** indicating whether the main answer is correct according to a trusted source

---

### Example

```json
[
    {
        "question_id": 34,
        "question": "What is the name of the low-cost carrier that operates as a wholly owned subsidiary of Singapore Airlines?",
        "answer": "Scoot is the low-cost carrier that operates as a wholly owned subsidiary of Singapore Airlines.",
        "tokens": [" Sco", "ot", " is", ..., " Airlines", ".", "\n"],
        "supporting_answers": [
            "As a wholly owned subsidiary of Singapore Airlines, <answer> Scoot </answer> stands as a low-cost carrier that revolutionized air travel in the region.",
            "Scoot, a subsidiary of <answer> Singapore Airlines </answer> , is the low-cost carrier that operates under the same brand.",
            "<answer> Scoot </answer> is the low-cost carrier that operates as a wholly owned subsidiary of Singapore Airlines.",
            "Singapore Airlines operates a low-cost subsidiary named <answer> Scoot </answer> , offering affordable and efficient air travel options to passengers."
        ],
        "supporting_tokens": [...],
        "supporting_probabilities": [...],
        "is_correct": true
    }
]
```

---

## Data

Available datasets for this task:

* `train.json` – contains 2967 questions and answers
* `valid.json` – contains 990 additional questions

---

## Evaluation Metric

**ROC AUC (Receiver Operating Characteristic Area Under Curve)** is a measure of binary classification quality. It reflects how well the model distinguishes between two classes — here: hallucination (**false**) vs. correct answer (**true**).

* **ROC Curve**: Shows the relationship between:

  * *True Positive Rate* (sensitivity)
  * *False Positive Rate* (1 - specificity)

* **AUC Score**:

  * **1.0** → perfect model
  * **0.5** → random model

The higher the AUC, the better the classifier.

---

## Scoring

You can earn between **0 and 100 points**, based on ROC AUC:

* **ROC AUC ≤ 0.7** → 0 points
* **ROC AUC ≥ 0.82** → 100 points
* Between 0.7 and 0.82 → linearly scaled

Formula:

$$
\text{Points} =
\begin{cases}
0 & \text{if } \text{ROC AUC} \leq 0.7 \\
100 \times \frac{\text{ROC AUC} - 0.7}{0.82 - 0.7} & \text{if } 0.7 < \text{ROC AUC} < 0.82 \\
100 & \text{if } \text{ROC AUC} \geq 0.82
\end{cases}
$$

---

## Constraints

* Your solution will be tested **without internet access** and **without GPU**
* Final evaluation must run **under 5 minutes**
* Allowed libraries:

  * `xgboost`
  * `scikit-learn`
  * `numpy`
  * `pandas`
  * `matplotlib`

---

## Submission Files

Submit the completed notebook with your solution implemented in the function:

```python
predict_hallucinations
```

---

## Final Evaluation

* During evaluation, `FINAL_EVALUATION_MODE = True`
* Your score is computed on a **hidden test set**
* Final score is **rounded to an integer**
* If your solution fails to run or violates constraints → **0 points**

## Starter Code

In this section, we initialize the environment by importing the necessary libraries and functions.
The provided code will help you efficiently work with the data and build an appropriate solution.


In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI PODCZAS WYSYŁANIA ##########################

FINAL_EVALUATION_MODE = False  # W czasie sprawdzania twojego rozwiązania, zmienimy tą wartość na True

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn as sk
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import shutil

def download_data(train=("1TGEDaxw4GKfSq0fpqSk0wRpUSc8GgZN0", "train.json"),
                  valid=("1qrr7bZk6Uct8DeC-V8Bc1qD5su56ryFd", "valid.json")):
    """Pobiera zbiór danych z Google Drive i zapisuje go w folderze 'data'."""
    import gdown
    
    # Utwórz lub zresetuj folder 'data'
    if not os.path.exists('data'):
        os.makedirs('data')
    else:
        shutil.rmtree('data')
        os.makedirs('data')

    GDRIVE_DATA = [train, valid]
    
    for file_id, file_name in GDRIVE_DATA:        
        # Pobierz plik z Google Drive i zapisz go w folderze 'data'
        url = f'https://drive.google.com/uc?id={file_id}'
        output = f'data/{file_name}'
        gdown.download(url, output, quiet=False)
        
        print(f"Downloaded: {file_name}")

# Pobierz dane tylko jeśli nie jesteś w trybie FINAL_EVALUATION_MODE
if not FINAL_EVALUATION_MODE:
    download_data()


## Data Loading

Using the code below, the data will be loaded and properly prepared.

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI PODCZAS WYSYŁANIA ##########################

def load_data(folder='data'):
    # Wczytaj dane z plików
    train_path = os.path.join(folder, 'train.json')
    valid_path = os.path.join(folder, 'valid.json')
    
    with open(train_path, 'r') as f:
        train = json.load(f)
    with open(valid_path, 'r') as f:
        valid = json.load(f)

    return train, valid

train, valid = load_data("data")

print(json.dumps(train[0], indent=2))

print(f"\nWszystkie przykłady treningowe: {len(train)}")
print(f"Wszystkie przykłady walidacyjne: {len(valid)}")

## Evaluation Code

Code similar to the one below will be used to evaluate your solution on the test dataset.

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI PODCZAS WYSYŁANIA ##########################

def compute_score(roc_auc: float) -> float:
    """
    Oblicza wynik punktowy na podstawie wartości ROC AUC.

    :param roc_auc: Wartość float w zakresie [0.0, 1.0]
    :return: Wynik punktowy zgodny z określoną funkcją
    """
    if roc_auc <= 0.7:
        return 0
    elif 0.7 < roc_auc < 0.82:
        return int(round(100 * (roc_auc - 0.7) / (0.82 - 0.7)))
    else:
        return 100


def evaluate_algorithm(dataset, algorithm, verbose=False):
    """
    Ewaluacja algorytmu wykrywania halucynacji na podanym zbiorze danych.

    Parametry
    ----------
    dataset : list
        Oznaczony zbiór danych, gdzie każdy element to słownik zawierający klucz 'is_correct'.
    algorithm : callable
        Funkcja, która przyjmuje pojedynczy przykład (słownik) i zwraca prawdopodobieństwo halucynacji.
    verbose : bool
        Jeśli True, wypisuje dodatkowe informacje dla każdego przykładu oraz podsumowanie.

    Zwraca
    -------
    roc_auc : float
        Wartość pola pod krzywą ROC (ROC AUC) dla predykcji.
    """
    predicted_ys = [] # Lista przechowująca przewidywane prawdopodobieństwa halucynacji

    for i, entry in enumerate(dataset):
        # Tworzenie kopii próbki i usunięcie etykiety, aby uzyskać dane wejściowe bez oznaczeń
        sample_unlabeled = dict(entry)
        sample_unlabeled.pop('is_correct', None)

        try:
            # Przewidywanie prawdopodobieństwa dla pojedynczej próbki
            pred_prob = algorithm(sample_unlabeled)
            predicted_ys.append(pred_prob)

        except Exception as e:
            # Jeśli wystąpi błąd, domyślnie ustawiamy prawdopodobieństwo na 0.5
            predicted_ys.append(0.5)
            if verbose:
                print(f"Sample {i} => Error: {e}")

    predicted_ys = np.array(predicted_ys, dtype=np.float32)
    ys = []
    for entry in dataset:
        ys.append(1 if entry.get('is_correct') else 0)
    ys = np.array(ys, dtype=np.int32)
    
    # Obliczenie metryki ROC AUC
    roc_auc = roc_auc_score(ys, predicted_ys)

    # Obliczenie końcowego wyniku na podstawie ROC AUC
    points = compute_score(roc_auc)

    if verbose:
        print(f"\nLiczba próbek: {len(dataset)}")
        print(f"ROC AUC: {roc_auc:.4f}")
        print(f"Wynik punktowy: {points}")

    return points

## Your Solution

In this section, you should place your solution.
Make changes **only here**!

In [ ]:
# TODO: Użyj danych treningowych, aby stworzyć tutaj model lub algorytm.

def predict_hallucinations(sample):
    # TODO: Uruchom swój model lub algorytm na tym zestawie danych.
    # TODO: Zwróć listę prawdopodobieństw dla każdego przykładu w zestawie danych.
    
    prediction = 0.5
    return prediction


## Evaluation

Running the cell below will allow you to check how many points your solution would achieve on the validation dataset.

Before submitting, make sure that the entire notebook runs from start to finish **without errors** and **without any user intervention** after selecting the **"Run All"** option.

In [ ]:
if not FINAL_EVALUATION_MODE:
    roc_auc = evaluate_algorithm(valid, predict_hallucinations, verbose=True)

During evaluation, your model will be saved as `your_model.pkl` and then assessed on the test dataset.

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI PODCZAS WYSYŁANIA ##########################
if FINAL_EVALUATION_MODE:      
    import cloudpickle
      
    OUTPUT_PATH = "file_output"
    FUNCTION_FILENAME = "your_model.pkl"
    FUNCTION_OUTPUT_PATH = os.path.join(OUTPUT_PATH, FUNCTION_FILENAME)

    if not os.path.exists(OUTPUT_PATH):
        os.makedirs(OUTPUT_PATH)

    with open(FUNCTION_OUTPUT_PATH, "wb") as f:
        cloudpickle.dump(predict_hallucinations, f)